# 03 — The transformation pipeline

The *Transform* step: going from heterogeneous raw publications to a clean, typed dataset
that conforms to the schema. The pipeline is laid out in three movements — **read**,
**process**, **export** — and every single transformation is a small, testable function.

In [1]:
import sys
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE / "src"))

from dotenv import load_dotenv

load_dotenv(RACINE / ".env")

from multimodal_etl.logging_setup import setup_logging

setup_logging()

## 1. The single-purpose functions

Each does one thing, which makes them readable and testable separately
(`tests/test_transform.py`).

In [2]:
from multimodal_etl.transform import (
    extract_domain,
    clean_text,
    normalise_date,
    normalise_label,
)

print(clean_text("<p>Un   résumé &amp; son <b>HTML</b></p>"))
print(extract_domain("https://www.bbc.co.uk/news/article-123"))
print(normalise_date("Mon, 29 Jun 2026 10:00:00 GMT"))
print(normalise_date("1767225600"))
print(normalise_label("FAKE"), normalise_label("Real"), normalise_label(""))

Un résumé & son HTML
bbc.co.uk
2026-06-29T10:00:00+00:00
2026-01-01T00:00:00+00:00
fake real None


Dates deserve a word: every source has its own format — RFC 822 for RSS, ISO for the API,
a Unix timestamp for Fakeddit. Without normalising, the freshness indicator would simply be
impossible to compute.

## 2. The rule that defines the dataset

`validate_image` does not look at the URL: it checks that **the file is on disk**. A
publication without an image does not enter the dataset. That check is what guarantees the
text-image pairing the use case demands.

In [3]:
from multimodal_etl.transform import validate_image

print(validate_image("data/raw/images/inexistante.jpg"))
print(validate_image(""))

False
False


## 3. Read → process → export

In [4]:
from multimodal_etl.config import RAW_DIR, TransformConfig
from multimodal_etl.transform import export_dataset, read_raw, process

config = TransformConfig()
dernier_brut = sorted(RAW_DIR.glob("raw_publications_*.json"))[-1]
brut = read_raw(dernier_brut)
print(f"{len(brut)} publications brutes lues")

2026-09-03 09:26:22 | INFO    | multimodal_etl.transform | Transform: reading raw file G:\Mon Drive\OC\portfolio\repos\lab-multimodal-etl-airflow\data\raw\raw_publications_20260903_072619.json


2026-09-03 09:26:22 | INFO    | multimodal_etl.transform | Transform: 164 raw publications read


164 publications brutes lues


In [5]:
df, stats = process(brut, config)
stats

2026-09-03 09:26:22 | INFO    | multimodal_etl.transform | Transform: 123/164 publications valid after cleaning


2026-09-03 09:26:22 | INFO    | multimodal_etl.transform | Transform: 0 duplicates removed


{'raw_total': 164,
 'valid_total': 123,
 'rejected': 41,
 'duplicates': 0,
 'with_image': 123,
 'labelled': 32,
 'native_images': 115,
 'open_graph_images': 8}

The rejection rate is high, and that is normal: almost every dropped publication goes for
a single reason — no usable image. Let us check that.

In [6]:
from multimodal_etl.transform import build_publication

raisons = {"titre vide": 0, "texte trop court": 0, "pas d'image": 0, "retenue": 0}
for publication in brut:
    if not clean_text(str(publication.get("title", ""))):
        raisons["titre vide"] += 1
    elif len(clean_text(str(publication.get("text", "")))) < config.min_text_length:
        raisons["texte trop court"] += 1
    elif not validate_image(str(publication.get("image_path", ""))):
        raisons["pas d'image"] += 1
    else:
        raisons["retenue"] += 1
raisons

{'titre vide': 0, 'texte trop court': 4, "pas d'image": 37, 'retenue': 123}

## 4. The dataset produced

The columns come straight from `multimodal_etl.schema`, the single source of truth shared
by the code, the diagram and the documentation.

In [7]:
df[["source", "access_method", "title", "image_source", "has_image", "label"]].head(8)

,source,access_method,title,image_source,has_image,label
0,rss:the_guardian,rss_feed,New constitution in Guinea-Bissau will undermi...,native,True,NaN
1,rss:the_guardian,rss_feed,Almost half of world’s farmers poisoned by pes...,native,True,NaN
2,rss:the_guardian,rss_feed,South African airline defends dramatic low-lev...,native,True,NaN
3,rss:the_guardian,rss_feed,Countries legally obliged to consider slavery ...,native,True,NaN
4,rss:the_guardian,rss_feed,"Morocco not to blame for Ceuta border breach, ...",native,True,NaN
5,rss:the_guardian,rss_feed,Apple Maps renames Lake Ontario as ‘Lake Ameri...,native,True,NaN
6,rss:the_guardian,rss_feed,Trump ally defends Venezuela oil deal amid ‘gu...,native,True,NaN
7,rss:the_guardian,rss_feed,Pete Hegseth criticized for body-shaming Canad...,native,True,NaN


In [8]:
print("Répartition par méthode d'accès :")
print(df["access_method"].value_counts().to_string())
print()
print("Origine des images :")
print(df["image_source"].value_counts().to_string())

Répartition par méthode d'accès :
access_method
rss_feed           91
kaggle_download    24
github_download     8

Origine des images :
image_source
native        115
open_graph      8


## 5. Export

The format chosen is **Parquet**: columnar, typed, compact. By this stage the schema is
fixed and the file is meant for analytical reads. The statistics are written next to it,
under the same name: the dashboard always loads the pair, never an orphan dataset.

In [9]:
path_for = export_dataset(df, config, stats)
print("Dataset :", path_for.name)
print("Statistiques :", path_for.with_name(path_for.stem + "_stats.json").name)

2026-09-03 09:26:23 | INFO    | multimodal_etl.transform | Transform: dataset of 123 rows exported to G:\Mon Drive\OC\portfolio\repos\lab-multimodal-etl-airflow\data\processed\publications_20260903_072623.parquet


2026-09-03 09:26:23 | INFO    | multimodal_etl.transform | Transform: statistics written to G:\Mon Drive\OC\portfolio\repos\lab-multimodal-etl-airflow\data\processed\publications_20260903_072623_stats.json


Dataset : publications_20260903_072623.parquet
Statistiques : publications_20260903_072623_stats.json
